# 맛 태그 키워드 정밀도 분석

**목적**: `filter_engine.TASTE_KEYWORDS`의 원재료 매칭 키워드 중 정밀도가 낮은(=거의 모든 과자에 매칭되는) 키워드를 데이터 기반으로 식별한다.

**룰**: 각 키워드가 전체 과자의 **30% 이상**에 매칭되면 노이즈로 간주하고 매칭 대상에서 제외.

**배경**: 이전에는 직관으로 'Strong/Weak'을 나누려 했으나 기준이 모호했음. 데이터 매칭률을 척도로 삼으면 분류가 객관화됨.

In [1]:
import sqlite3, sys, os
import pandas as pd

# backend/ 디렉토리 기준으로 동작하도록 sys.path 설정
ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.path.abspath('.')) != 'backend' else os.path.abspath('.')
if 'backend' not in ROOT:
    ROOT = os.path.abspath('.')
sys.path.insert(0, ROOT)

from app.services.filter_engine import TASTE_KEYWORDS

DB_PATH = os.path.join(ROOT, 'app', 'data', 'snack_products.sqlite3')
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT 품목명, 원재료명 FROM products', conn)
df['원재료명'] = df['원재료명'].fillna('')
df['품목명'] = df['품목명'].fillna('')
TOTAL = len(df)
print(f'전체 과자 수: {TOTAL}')

전체 과자 수: 540


## 1. 모든 원재료 키워드의 매칭률 측정
각 키워드가 데이터의 몇 %에 등장하는지 계산한다.

In [2]:
THRESHOLD = 0.30  # 30% 이상 매칭되는 키워드 = 노이즈

rows = []
for tag, keywords in TASTE_KEYWORDS.items():
    for kw in keywords:
        n = df['원재료명'].str.contains(kw, regex=False).sum()
        rate = n / TOTAL
        rows.append({
            '태그': tag,
            '키워드': kw,
            '매칭수': int(n),
            '매칭률': round(rate, 3),
            '판정': 'Strong (유지)' if rate < THRESHOLD else 'Weak (제외)',
        })

result_df = pd.DataFrame(rows).sort_values(['태그', '매칭률'], ascending=[True, False])
pd.set_option('display.max_rows', None)
result_df

,태그,키워드,매칭수,매칭률,판정
98,감자,감자,55,0.102,Strong (유지)
100,감자,감자전분,12,0.022,Strong (유지)
99,감자,감자분말,6,0.011,Strong (유지)
101,감자,감자플레이크,5,0.009,Strong (유지)
103,감자,건조감자분말,3,0.006,Strong (유지)
102,감자,감자그래뉼,1,0.002,Strong (유지)
104,고구마,고구마,6,0.011,Strong (유지)
106,고구마,고구마향,2,0.004,Strong (유지)
107,고구마,자색고구마,1,0.002,Strong (유지)
105,고구마,고구마분말,0,0.000,Strong (유지)


## 2. 제외 후보 키워드만 추출
30% 이상 매칭되어 노이즈로 분류된 키워드들. 이게 진짜 빠져야 하는 키워드 목록이다.

In [3]:
weak = result_df[result_df['판정'] == 'Weak (제외)'].sort_values('매칭률', ascending=False)
weak

,태그,키워드,매칭수,매칭률,판정
0,달달,설탕,343,0.635,Weak (제외)


## 3. 새 `TASTE_KEYWORDS` (자동 생성)
Strong 키워드만 남긴 dict. 이걸 그대로 `filter_engine.py`에 붙여넣으면 된다.

In [4]:
new_kw = {}
for tag, keywords in TASTE_KEYWORDS.items():
    survivors = []
    for kw in keywords:
        rate = df['원재료명'].str.contains(kw, regex=False).sum() / TOTAL
        if rate < THRESHOLD:
            survivors.append(kw)
    if survivors:
        new_kw[tag] = survivors

print('TASTE_KEYWORDS = {')
for tag, kws in new_kw.items():
    kw_repr = ', '.join(f'"{k}"' for k in kws)
    print(f'    "{tag}": [{kw_repr}],')
print('}')
print()
removed_tags = set(TASTE_KEYWORDS) - set(new_kw)
print(f'(완전히 제거된 태그: {sorted(removed_tags)})')

TASTE_KEYWORDS = {
    "달달": ["백설탕", "갈색설탕", "흑설탕", "물엿", "올리고당", "꿀", "벌꿀", "허니", "달콤", "달고나", "단밤", "바닐라", "바닐린", "카스타드", "크림", "생크림"],
    "카라멜": ["카라멜", "캐러멜", "골든시럽", "단풍당시럽", "당밀시럽"],
    "초코": ["초콜릿", "초코", "카카오", "코코아", "코코아매스", "다크초콜릿", "밀크초콜릿", "화이트초콜릿", "준초콜릿", "초코칩", "코코아분말"],
    "말차": ["말차", "녹차", "녹차가루", "그린티"],
    "커피": ["커피", "카푸치노", "모카", "커피향", "라떼"],
    "버터갈릭": ["버터", "가공버터", "무염버터", "버터향", "갈릭", "마늘", "마늘가루", "마늘페이스트", "건마늘분말"],
    "치즈": ["치즈", "체다치즈", "치즈분말", "가공치즈", "크림치즈", "치즈향", "체다치즈분말"],
    "고소": ["참깨", "볶은참깨", "검은깨", "흑임자", "참기름", "들기름", "볶음땅콩", "아몬드", "호두", "잣", "피스타치오"],
    "매콤": ["고추", "고춧가루", "청양고추", "스리라차", "할라피뇨", "매운", "칠리", "페퍼"],
    "바베큐": ["바베큐", "바비큐", "숯불", "훈제", "스모키"],
    "양파": ["양파", "양파분말", "양파향", "어니언", "볶음양파분말"],
    "와사비": ["와사비", "고추냉이"],
    "새우": ["새우", "냉동새우", "새우분말", "새우엑기스", "새우맛씨즈닝"],
    "오징어": ["오징어", "오징어엑기스", "오징어페이스트", "버터구이오징어"],
    "감자": ["감자", "감자분말", "감자전분", "감자플레이크", "감자그래뉼", "건조감자분말"],
    "고구마": ["고구마", "고구마분말", 

## 4. 검증 — 사용자가 지적한 케이스들
새 키워드 사전으로 다시 태깅했을 때 결과가 의도대로 나오는지 확인.

In [5]:
from app.services.filter_engine import NAME_TASTE_KEYWORDS

def tag_new(ingredients, name):
    tags = set()
    for tag, kws in new_kw.items():
        for kw in kws:
            if kw in ingredients:
                tags.add(tag)
                break
    for tag, kws in NAME_TASTE_KEYWORDS.items():
        for kw in kws:
            if kw in name:
                tags.add(tag)
                break
    return sorted(tags)

targets = ['미쯔', '계란과자', '웨하스', '스윗', '솔티', '꼬북칩', '치토스']
for _, r in df.iterrows():
    name = r['품목명']
    if any(t in name for t in targets):
        print(f'  {name}  ::  {tag_new(r["원재료명"], name)}')

  서주 허쉬 솔티카라멜 와플  ::  ['달달', '버터갈릭', '초코', '카라멜']
  서주 포켓 웨하스 그릭요거트  ::  ['사과', '요거트']
  빼빼로 더블리치 솔티바닐라  ::  ['달달', '버터갈릭', '초코']
  서주 미니 웨하스 바닐라맛  ::  ['달달']
  서주 미니 웨하스 딸기맛  ::  ['딸기']
  꼬북칩 초코츄러스맛  ::  ['버터갈릭', '초코']
  삼아 그린티 웨하스  ::  ['말차']
  계란과자  ::  ['달달', '버터갈릭']
  포카칩 스윗치즈맛  ::  ['감자', '치즈']
  치토스 스모키바베큐맛  ::  ['바베큐']
  삼아 우유 웨하스  ::  ['달달']
  삼아 미니 우리밀 웨하스  ::  ['달달']
  엄마손파이 스윗버터갈릭  ::  ['버터갈릭']
  솔티드 라지 프레첼  ::  []
  서주딸기웨하스  ::  ['딸기', '초코']
  서주 포켓 웨하스 초코바나나  ::  ['바나나', '초코']
  서주 포켓 웨하스 초코  ::  ['달달', '버터갈릭', '초코']
  서주 미니 베리베리스트로베리 웨하스  ::  []
  미쯔블랙  ::  ['달달', '초코']
  삼아 미니 바닐라 웨하스  ::  ['달달']
  꼬북칩 콘스프맛  ::  ['옥수수']
  꼬북칩 카라멜팝콘맛  ::  ['달달', '옥수수', '카라멜']
  꼬북칩 초코츄러스맛  ::  ['달달', '버터갈릭', '초코']
  꼬북칩 말차초코맛  ::  ['말차', '버터갈릭', '초코']
  계란과자 치즈  ::  ['치즈']
  해태 딸기웨하스  ::  ['딸기']
  서주 홈앤키즈 칼슘웨하스 딸기  ::  ['달달', '딸기']
  서주 홈앤키즈 초유웨하스 바닐라  ::  []
  삼아 미니 피스타치오 웨하스  ::  ['고소']
  플레인계란과자  ::  ['달달', '버터갈릭']
  서주 티젠 티오마카세 호지차 웨하스  ::  []
  서주 티젠 티오마카세 얼그레이 웨하스  ::  []
  서주 티젠 티

## 5. 옥수수+짭짤 / 옥수수+달달 결과 미리보기

In [6]:
def match_tastes_new(tags, selected):
    FRUIT = {'딸기','바나나','복숭아','사과','파인애플','멜론','블루베리','귤감귤'}
    if not selected:
        return True
    tag_set = set(tags)
    fruit_sel = [t for t in selected if t in FRUIT]
    non_fruit = [t for t in selected if t not in FRUIT]
    if fruit_sel and not any(t in tag_set for t in fruit_sel):
        return False
    if not all(t in tag_set for t in non_fruit):
        return False
    return True

all_tags = df.apply(lambda r: tag_new(r['원재료명'], r['품목명']), axis=1)
df_t = df.copy()
df_t['tags'] = all_tags

for combo in [['옥수수','짭짤'], ['옥수수','달달'], ['옥수수','매콤'], ['달달','초코']]:
    hits = df_t[df_t['tags'].apply(lambda t: match_tastes_new(t, combo))]
    print(f'=== {combo} — {len(hits)}건 ===')
    for _, r in hits.head(8).iterrows():
        print(f'  {r["품목명"]}  ::  {r["tags"]}')
    print()

=== ['옥수수', '짭짤'] — 2건 ===
  오리지널 콘칩  ::  ['옥수수', '짭짤']
  소금우유맛팝콘  ::  ['옥수수', '짭짤']

=== ['옥수수', '달달'] — 17건 ===
  핫고래밥 매콤양념맛  ::  ['달달', '매콤', '옥수수']
  옥수수 에이플러스콘  ::  ['달달', '옥수수']
  고래밥 볶음양념맛  ::  ['달달', '옥수수']
  옥수수 고추맛콘  ::  ['달달', '매콤', '옥수수']
  옥수수 미니콘  ::  ['달달', '옥수수']
  밀크카라멜팝콘  ::  ['달달', '옥수수', '카라멜']
  꼬북칩 카라멜팝콘맛  ::  ['달달', '옥수수', '카라멜']
  허니버터 팝콘  ::  ['달달', '버터갈릭', '옥수수']

=== ['옥수수', '매콤'] — 5건 ===
  핫고래밥 매콤양념맛  ::  ['달달', '매콤', '옥수수']
  옥수수 고추맛콘  ::  ['달달', '매콤', '옥수수']
  칠리버터맛 콘칩  ::  ['매콤', '버터갈릭', '옥수수']
  매운맛콘칩  ::  ['매콤', '옥수수']
  짜장라면맛 고래밥  ::  ['달달', '매콤', '옥수수']

=== ['달달', '초코'] — 65건 ===
  통크 초코  ::  ['달달', '초코']
  쿠크슈몬 화이트  ::  ['달달', '버터갈릭', '초코', '치즈']
  쵸코하임  ::  ['달달', '초코']
  제로 쿠앤크 샌드  ::  ['달달', '초코']
  자주 꾀돌이  ::  ['달달', '초코']
  서주 허쉬 헤이즐넛 와플  ::  ['달달', '버터갈릭', '초코']
  서주 허쉬 솔티카라멜 와플  ::  ['달달', '버터갈릭', '초코', '카라멜']
  빼빼로 더블리치 솔티바닐라  ::  ['달달', '버터갈릭', '초코']



## 다음 단계

위 §3의 출력을 [filter_engine.py](app/services/filter_engine.py)의 `TASTE_KEYWORDS`에 적용하면 30% 임계치 기반의 정밀한 태깅이 활성화된다.

원하면 임계치(`THRESHOLD`)를 0.20/0.50 등으로 바꿔서 재실행하여 비교 가능.